In [24]:
# imports
import pandas as pd
import torch
from torch import nn

%load_ext autoreload
%autoreload 2
import models as m
import evaluation as ev

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ImportError: dlopen(/Users/mds/Documents/MSc/10_Project/product-sales/.venv/lib/python3.10/site-packages/scipy/sparse/linalg/_propack/_spropack.cpython-310-darwin.so, 0x0002): tried: '/Users/mds/Documents/MSc/10_Project/product-sales/.venv/lib/python3.10/site-packages/scipy/sparse/linalg/_propack/_spropack.cpython-310-darwin.so' (section '__DATA/__thread_bss' has a zero-fill section type, but offset field is not zero), '/System/Volumes/Preboot/Cryptexes/OS/Users/mds/Documents/MSc/10_Project/product-sales/.venv/lib/python3.10/site-packages/scipy/sparse/linalg/_propack/_spropack.cpython-310-darwin.so' (no such file), '/Users/mds/Documents/MSc/10_Project/product-sales/.venv/lib/python3.10/site-packages/scipy/sparse/linalg/_propack/_spropack.cpython-310-darwin.so' (section '__DATA/__thread_bss' has a zero-fill section type, but offset field is not zero)

In [19]:
if torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

In [9]:
fold_labels = pd.read_csv('data/fold_labels.csv')
img_parquet  = pd.read_parquet('data/clip_image.parquet')
txt_parquet  = pd.read_parquet('data/clip_text.parquet')

label_keys = set(fold_labels['external_code'])
missing_img = label_keys - set(img_parquet['external_code'])
missing_txt = label_keys - set(txt_parquet['external_code'])

# merge should have 5080 products
print("Fold labels shape:", fold_labels.shape)
print("Img parquet shape:", img_parquet.shape)
print("Text parquet shape:", txt_parquet.shape)
print("Missing img?", len(missing_img))
print("Missing txt?", len(missing_txt))
print("External code unique?", fold_labels['external_code'].nunique())


Fold labels shape: (20341, 5)
Img parquet shape: (5577, 769)
Text parquet shape: (5577, 769)
Missing img? 0
Missing txt? 0
External code unique? 5080


In [10]:
# index both towers by external code
txt_embeddings = txt_parquet.set_index('external_code')
img_embeddings = img_parquet.set_index('external_code')

is_fold_0_train = (fold_labels['fold'] == 0) & (fold_labels['split'] == 'train')
fold_0_train = fold_labels[is_fold_0_train]

train_codes = fold_0_train['external_code'].to_numpy()

train_txt_x = txt_embeddings.loc[train_codes]
train_img_x = img_embeddings.loc[train_codes]
train_labels = fold_0_train['label'].to_numpy()

print("Fold 0 shape:", fold_0_train.shape)
print("Text X shape", train_txt_x.shape)
print("Image X shape", train_img_x.shape)
print("Labels Length:", len(train_labels))

print((train_txt_x.index.to_numpy() == train_codes).all())
print((train_img_x.index.to_numpy() == train_codes).all())

Fold 0 shape: (3053, 5)
Text X shape (3053, 768)
Image X shape (3053, 768)
Labels Length: 3053
True
True


In [11]:
# index both towers by external code
is_fold_0_val = (fold_labels['fold'] == 0) & (fold_labels['split'] == 'val')
fold_0_val = fold_labels[is_fold_0_val]

val_codes = fold_0_val['external_code'].to_numpy()

train_labels = fold_0_val['label'].to_numpy()
in_buffer = fold_0_val['in_buffer'].to_numpy()

print("Non buffer:", (~in_buffer).sum())
print(fold_0_val['label'].value_counts().sort_index())

Non buffer: 769
label
0    253
1    509
2    254
Name: count, dtype: int64


In [12]:
# return fold's data for one split, aligned by product
def get_fold_data(fold, split):
    selected_rows = (fold_labels['fold'] == fold) & (fold_labels['split'] == split)
    fold_rows = fold_labels[selected_rows]

    codes = fold_rows['external_code'].to_numpy()

    fold_data = {
        'text': txt_embeddings.loc[codes],
        'image': img_embeddings.loc[codes],
        'labels': fold_rows['label'].to_numpy(),
        'in_buffer': fold_rows['in_buffer'].to_numpy(),
        'codes': codes,
    }

    return fold_data

print(get_fold_data(3, 'val')['text'].shape[0])
print((~get_fold_data(3, 'val')['in_buffer']).sum())

1016
772


In [13]:
# function to turn the fold data into tensors for pytorch

def to_tensors(fold_data, device):
    text_tensor = torch.from_numpy(fold_data['text'].values).to(device)
    image_tensor = torch.from_numpy(fold_data['image'].values).to(device)

    labels_tensor = torch.as_tensor(fold_data['labels'], dtype=torch.int64).to(device)

    in_buffer = fold_data['in_buffer']
    codes = fold_data['codes']

    return {
        'text': text_tensor,
        'image': image_tensor,
        'labels': labels_tensor,
        'in_buffer': in_buffer,
        'codes': codes
    }

train_data = to_tensors(get_fold_data(0, 'train'), device)
val_data = to_tensors(get_fold_data(0, 'val'), device)

print(train_data['text'].shape, train_data['text'].dtype, train_data['text'].device)
print(train_data['image'].shape, train_data['image'].dtype, train_data['image'].device)
print(train_data['labels'].shape, train_data['labels'].dtype, train_data['labels'].device)
print(train_data['in_buffer'].dtype, len(train_data['in_buffer']), train_data['in_buffer'].sum())
print(train_data['codes'].dtype, len(train_data['codes']))


torch.Size([3053, 768]) torch.float32 mps:0
torch.Size([3053, 768]) torch.float32 mps:0
torch.Size([3053]) torch.int64 mps:0
bool 3053 0
int64 3053


In [14]:
dropout = 0.2
n_classes = 3
batch_size = 64
learning_rate = 1e-3
max_epochs = 50
seed = 7

def train_one_run (condition, hidden_width, fold, train_data, val_data):
    torch.manual_seed(seed)

    epoch_rows = []
    input_dim = train_data['text'].shape[1]

    config = {
        'condition': condition,
        'input_dim': input_dim,
        'hidden_width': hidden_width,
        'dropout': dropout,
        'n_classes': n_classes,
    }

    model = m.ConditionModel(config).to(train_data['text'].device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=0,
        betas=(0.9, 0.999),
        eps=1e-8,
    )

    loss_function = nn.CrossEntropyLoss()
    n_train = train_data['text'].shape[0]

    for epoch in range(1, max_epochs +1):
        model.train()
        running_loss = 0.0

        shuffled = torch.randperm(n_train, device=train_data['text'].device)

        for start in range(0, n_train, batch_size):
            batch_rows = shuffled[start:start + batch_size]

            batch_text = train_data['text'][batch_rows]
            batch_image = train_data['image'][batch_rows]
            batch_labels = train_data['labels'][batch_rows]

            optimizer.zero_grad()
            logits = model(batch_text, batch_image)
            loss = loss_function(logits, batch_labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(batch_rows)

        train_loss = running_loss / n_train

        model.eval()
        with torch.no_grad():
            val_logits = model(val_data['text'], val_data['image'])

        predictions = val_logits.argmax(dim=1).cpu().numpy()

        scores = m.score_fold(
            val_data['labels'].cpu().numpy(),
            predictions,
            val_data['in_buffer'],
        )

        epoch_rows.append({
            'condition': condition,
            'hidden_width': hidden_width,
            'fold': fold,
            'epoch': epoch,
            'train_loss': train_loss,
            'balanced_accuracy_full': scores['balanced_accuracy_full']
        })

    return epoch_rows

In [15]:
check_model = m.ConditionModel({
    'condition': 'text',
    'input_dim': 768,
    'hidden_width': 256,
    'dropout': 0.2,
    'n_classes': 3
})

sum(p.numel() for p in check_model.parameters())

263427

In [16]:
fold_0_train = to_tensors(get_fold_data(0, 'train'), device)
fold_0_val = to_tensors(get_fold_data(0, 'val'), device)

text_rows = train_one_run('text', 256, 0, fold_0_train, fold_0_val)

curve = pd.DataFrame(text_rows)
curve.head(10)
curve.tail(5)

AttributeError: module 'models' has no attribute 'score_fold'

In [23]:
import numpy, scipy, sklearn
print(numpy.__version__, scipy.__version__, sklearn.__version__)

ImportError: dlopen(/Users/mds/Documents/MSc/10_Project/product-sales/.venv/lib/python3.10/site-packages/scipy/sparse/linalg/_propack/_spropack.cpython-310-darwin.so, 0x0002): tried: '/Users/mds/Documents/MSc/10_Project/product-sales/.venv/lib/python3.10/site-packages/scipy/sparse/linalg/_propack/_spropack.cpython-310-darwin.so' (section '__DATA/__thread_bss' has a zero-fill section type, but offset field is not zero), '/System/Volumes/Preboot/Cryptexes/OS/Users/mds/Documents/MSc/10_Project/product-sales/.venv/lib/python3.10/site-packages/scipy/sparse/linalg/_propack/_spropack.cpython-310-darwin.so' (no such file), '/Users/mds/Documents/MSc/10_Project/product-sales/.venv/lib/python3.10/site-packages/scipy/sparse/linalg/_propack/_spropack.cpython-310-darwin.so' (section '__DATA/__thread_bss' has a zero-fill section type, but offset field is not zero)